# NPTEL Study Notes: Knowledge Graph Embeddings (Factor & Multiplicative Models)

**Course Context:** While part of a broader curriculum on Large Language Models (LLMs) and NLP, this specific module focuses on **Knowledge Graph (KG) Embeddings**, specifically transitioning from translation/rotation models to **multiplicative (factor) models**.

---

## 1. Background: Matrix Factorization & SVD

Multiplicative models aim to predict a property $y_{ij}$ of two items $i$ and $j$ by computing the inner product of their vector representations (featurizations): $y_{ij} = f(i) \cdot f(j)$. Historically, this stems from Latent Semantic Analysis (1988) and forms the basis of Word2Vec/GloVe.

### The Document-Term Matrix

Consider a corpus represented as a matrix $M$ where rows are documents and columns are words.

* **Values:** Can be Boolean (1 if word exists in doc, 0 otherwise) or frequency-based (count of word in doc).
* **Duality of Similarity:** Documents are similar if they share similar words. Words are similar if they appear in similar documents.

### Singular Value Decomposition (SVD)

SVD factors any linear transformation matrix $M$ into three matrices:


$$M = U \Sigma V^T$$

* **$U$ Matrix:** Represents documents. Columns are orthogonal unit vectors ($L2$ norm = 1).
* **$\Sigma$ Matrix:** A diagonal matrix containing singular values ($\sigma_1 \ge \sigma_2 \dots \ge 0$).
* **$V^T$ Matrix:** Represents words. Rows are orthogonal unit vectors.

**Low-Rank Approximation (Rank Revealing):**
By setting the smaller singular values in $\Sigma$ to 0 (and dropping the corresponding rows/columns in $U$ and $V^T$), we create a low-rank approximation of $M$ that filters out noise. The truncated columns of $V^T$ yield dense **word embeddings**.

**Computing Similarity:** To find the similarity between two words (e.g., "cat" and "dog"), we approximate their co-occurrence by taking the inner product of their corresponding vector representations extracted from SVD. SVD provides noise reduction, yielding a smooth similarity score.

---

## 2. Knowledge Graphs as 3D Tensors

A Knowledge Graph represents facts as triples: **(Subject, Relation, Object)**.
We can represent the entire KG as a 3D Boolean Tensor $X$ with three axes:

1. **Subject ($S$)**
2. **Object ($O$)**
3. **Relation ($R$)**

**Tensor Properties:**

* **Shape:** $E \times E \times R$ (where $E$ is the number of entities, $R$ is the number of relations).
* **Values:** $X_{SRO} = 1$ if the fact is true in the KG, $0$ otherwise.
* **Sparsity & Low Rank:** $X$ is extremely sparse (very few possible facts are actually true). It is also strictly low-rank because entities share similarities (e.g., CEOs are also employees) and relations are highly correlated (e.g., "born in" strongly correlates with "citizen of").

---

## 3. 3D Tensor Factorization

Just as we factored the 2D document-term matrix, we want to factor the 3D tensor $X$ into lower-dimensional embeddings:


$$X \approx A \times M \times A'$$

* **$A$ and $A'$:** Entity embedding matrices of size $E \times D$ (where $D$ is the embedding dimension).
* **$M$:** A relation tensor of size $D \times D \times R$. Each relation $r$ gets a $D \times D$ matrix slice, $M_r$.

**Optimization Objective:**
We want to find $A$ and $M$ that minimize the difference between the actual tensor $X$ and our approximation. We use the **Frobenius norm** (sum of squared errors element-wise) over all $(S, R, O)$ combinations.

To prevent values from exploding, we add regularization terms (Frobenius norms of $A$ and $M$). Optimization is typically done via **alternating sub-steps**: fix $M$ to update $A$ via gradient descent, then fix $A$ to update $M$.
*Note: Unlike SVD, this is not guaranteed to find a unique global optimum; it finds a local optimum.*

---

## 4. Addressing Tensor Factorization Complexities

Optimizing a full $D \times D$ matrix for every relation is computationally expensive. Researchers developed two main ways to simplify this:

### Technique 1: Flattening to a 2D Matrix

Group $(S, O)$ pairs into rows ($E^2$ rows) and Relations ($R$) into columns. We get an $E^2 \times R$ Boolean matrix, which can be factored using SVD.

* **Drawback:** You only learn embeddings for *seen entity pairs*. If a new entity arrives, you cannot form an individual vector for it.

### Technique 2: Simplifying the Relation Tensor (DistMult)

Instead of a full $D \times D$ matrix for each relation $M_r$, we force $M_r$ to be a **diagonal matrix** (off-diagonals are 0).

This reduces the scoring function to a simple **3-way inner product** of the vectors:


$$Score(S, R, O) \approx \langle S, R, O \rangle = \sum_{d=1}^{D} s_d \cdot r_d \cdot o_d$$

| Model | Advantage | Limitation |
| --- | --- | --- |
| **DistMult** | Extremely fast and easy to train (reduces complexity from $D^2$ to $D$ per relation). Performs well on basic benchmarks. | **Strictly Symmetric:** Because multiplication is commutative ($S \cdot R \cdot O = O \cdot R \cdot S$), it cannot model asymmetric relations (e.g., "follows on social media") or anti-symmetric relations (e.g., "father of"). |

*Why did DistMult appear to perform well initially?* Datasets like FB15K rarely queried the inverse (symmetric) direction in the test set (only 4% of queries), hiding DistMult's architectural flaw.

---

## 5. Advanced Multiplicative Models

To fix the symmetric limitations of DistMult, two major extensions were created:

### A. SimplE

SimplE addresses asymmetry by giving each entity **two separate vectors**: one when it acts as a Subject ($sub_e$) and one when it acts as an Object ($obj_e$).

* It also automatically introduces an **inverse relation ($R^{-1}$)** for every relation $R$.
* **Scoring Function:** Averages the forward and backward scores:

$$Score = \frac{1}{2} ( \langle sub_s, R, obj_o \rangle + \langle sub_o, R^{-1}, obj_s \rangle )$$


* **Result:** Capable of modeling symmetric, asymmetric, and anti-symmetric relations perfectly if $D$ is large enough.

### B. ComplEx

ComplEx uses **complex numbers** (with real and imaginary parts) for embeddings instead of real numbers.

* The scoring function takes the **Real part** of the inner product, where the object vector is conjugated ($O^*$):

$$Score = \text{Re}(\langle S, R, O^* \rangle)$$


* **Handling Symmetry:** Set the imaginary part of $R$ to 0 ($\text{Im}(R) = 0$). This reduces mathematically to DistMult.
* **Handling Anti-Symmetry:** Set the real part of $R$ to 0 ($\text{Re}(R) = 0$). Flipping the arguments negates the score, perfectly modeling anti-symmetry (if $S \to O$ is highly true, $O \to S$ becomes highly false).

### C. ComplEx-N3

Combines features of both. It uses complex vectors, two representations per entity (Subject/Object), introduces $R^{-1}$, and regularizes using the **nuclear p-norm** (rather than standard L2/Frobenius). ComplEx-N3 is currently among the most competitive KG completion algorithms.

---

## 6. Benchmarks and Data Leakage

Evaluating these models requires datasets. Two classic benchmarks had to be updated due to **data leakage**:

1. **WN18 (WordNet) $\rightarrow$ WN18RR:** WN18 had severe leakage where test triples often had their exact inverse present in the training set. Models could "cheat" just by memorizing inverses. WN18RR removed these to create a harder, more accurate benchmark (reduced to 11 relation types).
2. **FB15K (Freebase) $\rightarrow$ FB15K-237:** Followed a similar cleanup, removing inverse triple leakages and dropping the relation count from 1,345 to 237.

---

## Practice Questions

**Q1: In the standard SVD factorization $M = U \Sigma V^T$, what does the matrix $\Sigma$ represent and what is its structure?**
**Answer:** $\Sigma$ is a diagonal matrix containing the singular values of $M$, sorted in descending order ($\sigma_1 \ge \sigma_2 \ge \dots \ge 0$). The off-diagonal elements are all zeros.

**Q2: Why is the DistMult model fundamentally unable to accurately represent the relationship "is the mother of"?**
**Answer:** "Is the mother of" is an *anti-symmetric* relation (if A is the mother of B, B cannot be the mother of A). DistMult relies on a standard dot product $\langle S, R, O \rangle$, which is commutative. Therefore, DistMult will incorrectly assign the exact same truth score to both $(A, \text{mother\_of}, B)$ and $(B, \text{mother\_of}, A)$.

**Q3: How does the ComplEx model configure its relation vector to specifically handle an anti-symmetric relation?**
**Answer:** In ComplEx, an anti-symmetric relation is modeled by constraining the real part of the relation vector $R$ to zero ($\text{Re}(R) = 0$). This mathematical constraint ensures that reversing the subject and object negates the overall real score of the triple.

**Q4: Why was the WN18 benchmark dataset replaced by WN18RR for evaluating knowledge graph completion?**
**Answer:** WN18 suffered from data leakage. Many facts in the test set had their exact inverse triples present in the training data. This allowed rudimentary algorithms to overfit and artificially achieve high scores simply by memorizing inverse patterns rather than learning true graph embeddings. WN18RR removed these leaking triples.